## Example 1: Write values and formats, read them back with openpyxl

In [ ]:
import importlib.metadata
import io

import openpyxl
import xlsxwriter

# Example 1: strings, numbers and a bold header format survive a round trip through openpyxl
assert importlib.metadata.version("xlsxwriter") == "3.3.1"
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    ws = wb.add_worksheet("data")
    ws.write_row(0, 0, ["city", "pop"], wb.add_format({"bold": True}))
    ws.write_row(1, 0, ["Oslo", 709037])
ws2 = openpyxl.load_workbook(io.BytesIO(buf.getvalue()))["data"]
assert [c.value for c in ws2[2]] == ["Oslo", 709037] and ws2["A1"].font.bold is True
print("Example 1 OK:", [[c.value for c in row] for row in ws2.iter_rows()])

## Example 2: Formulas keep the formula and a cached result

In [ ]:
import io

import openpyxl
import xlsxwriter

# Example 2: a formula cell stores its text; xlsxwriter can also store the value a reader shows without Excel
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    ws = wb.add_worksheet()
    ws.write_column("A1", [1, 2, 3])
    ws.write_formula("A4", "=SUM(A1:A3)", None, 6)
book = io.BytesIO(buf.getvalue())
assert openpyxl.load_workbook(book)["Sheet1"]["A4"].value == "=SUM(A1:A3)"
assert openpyxl.load_workbook(book, data_only=True)["Sheet1"]["A4"].value == 6
print("Example 2 OK")

## Example 3: A native Excel chart

In [ ]:
import io
import zipfile

import xlsxwriter

# Example 3: charts are written as DrawingML parts inside the .xlsx zip
buf = io.BytesIO()
with xlsxwriter.Workbook(buf) as wb:
    ws = wb.add_worksheet()
    ws.write_column("A1", [3, 1, 4, 1, 5])
    chart = wb.add_chart({"type": "column"})
    chart.add_series({"values": "=Sheet1!$A$1:$A$5", "name": "pi digits"})
    ws.insert_chart("C2", chart)
parts = zipfile.ZipFile(io.BytesIO(buf.getvalue())).namelist()
assert "xl/charts/chart1.xml" in parts and "xl/drawings/drawing1.xml" in parts, parts
print("Example 3 OK:", sorted(p for p in parts if "chart" in p))

## Example 4: pandas.to_excel(engine='xlsxwriter') read back by calamine

In [ ]:
import io

import pandas as pd

# Example 4: pandas writes through xlsxwriter; pandas' calamine engine reads the file back
df = pd.DataFrame({"name": ["ada", "bob"], "score": [9.5, 7.0], "passed": [True, False]})
buf = io.BytesIO()
with pd.ExcelWriter(buf, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="results", index=False)
back = pd.read_excel(io.BytesIO(buf.getvalue()), sheet_name="results", engine="calamine")
assert back.equals(df), back
print("Example 4 OK:\n", back)

## Example 5: polars.write_excel and constant-memory mode

In [ ]:
import io

import polars as pl
import xlsxwriter

# Example 5: polars' write_excel uses xlsxwriter; constant_memory streams rows to disk as they are written
df = pl.DataFrame({"x": [1, 2, 3], "y": ["a", "b", "c"]})
buf = io.BytesIO()
df.write_excel(buf, worksheet="pl")
assert pl.read_excel(io.BytesIO(buf.getvalue()), sheet_name="pl").equals(df)
big = io.BytesIO()
with xlsxwriter.Workbook(big, {"constant_memory": True, "in_memory": False}) as wb:
    ws = wb.add_worksheet()
    for r in range(2000):
        ws.write_row(r, 0, [r, r * r])
assert pl.read_excel(io.BytesIO(big.getvalue()), has_header=False).height == 2000
print("Example 5 OK")